<a href="https://colab.research.google.com/github/agusbass/ARSI/blob/main/ARSI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARSI — Semantic Drift Monitoring for AI Systems

**ARSI** (Adaptive Representation & Structure Interface) is a lightweight system for detecting topic drift (semantic drift) in conversations or text.

> 🔍 **ARSI is not just cosine similarity.**  
> It adds:
> - Adaptive semantic fields (spread & radius per topic)
> - Per‑field temporal memory (drift per topic, not mixed)
> - Probabilistic routing with temperature scaling
> - Combined certainty score (confidence, margin, entropy)

📌 **Status:** Experimental research prototype. Suitable for observability, debugging, and experimentation.

In [ ]:
!pip install sentence-transformers scipy matplotlib pandas -q
print("✓ Installation complete")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy
import pickle
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"✓ Model loaded, dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
TEMPERATURE = 0.12
EPS = 1e-8
LEARNING_RATE = 0.05

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + EPS))

def softmax_temp(x, T=TEMPERATURE):
    x = np.array(x)
    scaled = x / T
    e = np.exp(scaled - np.max(scaled))
    return e / np.sum(e)

print("✓ Utility functions ready")

In [ ]:
class AdaptiveField:
    def __init__(self, name, center, spread=0.1, radius=0.05):
        self.name = name
        self.center = center.copy()
        self.spread = spread
        self.radius = radius
        self.temporal_memory = None
        self.interaction_count = 0

    def update(self, new_vector, lr=LEARNING_RATE):
        self.center = (1 - lr) * self.center + lr * new_vector
        self.interaction_count += 1

    def calibrated_similarity(self, vec):
        sim = cosine_sim(self.center, vec)
        return sim * np.exp(-self.spread)

class AdaptiveFieldSystem:
    def __init__(self, fields_dict, alpha=0.8):
        self.fields = fields_dict
        self.alpha = alpha
        self.reset_session()

    def reset_session(self):
        for f in self.fields.values():
            f.temporal_memory = None

    def add_interaction_feedback(self, field_name, corrected_vector):
        if field_name in self.fields:
            self.fields[field_name].update(corrected_vector)

    def check(self, text):
        vec = model.encode(text)
        sims, calib = {}, {}
        for name, field in self.fields.items():
            s = cosine_sim(field.center, vec)
            sims[name] = s
            calib[name] = field.calibrated_similarity(vec)

        score_vals = np.array(list(calib.values()))
        probs = softmax_temp(score_vals)
        field_names = list(calib.keys())
        best_idx = np.argmax(probs)
        best_field = field_names[best_idx]
        best_prob = probs[best_idx]
        best_sim = sims[best_field]

        sorted_probs = np.sort(probs)[::-1]
        margin = sorted_probs[0] - sorted_probs[1] if len(sorted_probs) > 1 else 1.0
        ent = entropy(probs) / np.log(len(probs)) if len(probs) > 1 else 0.0

        drift = 1.0 - best_sim

        mem = self.fields[best_field].temporal_memory
        if mem is None:
            self.fields[best_field].temporal_memory = drift
        else:
            self.fields[best_field].temporal_memory = self.alpha * mem + (1 - self.alpha) * drift
        temporal_drift = self.fields[best_field].temporal_memory

        certainty = 0.4 * best_prob + 0.3 * margin + 0.3 * (1 - ent)
        certainty = np.clip(certainty, 0.0, 1.0)

        return {
            "drift": round(drift, 4),
            "temporal_drift": round(temporal_drift, 4),
            "similarity": round(best_sim, 4),
            "best_field": best_field,
            "confidence": round(best_prob, 4),
            "margin": round(margin, 4),
            "entropy": round(ent, 4),
            "certainty": round(certainty, 4),
            "probabilities": {k: round(v, 4) for k, v in zip(field_names, probs)}
        }

print("✓ AdaptiveFieldSystem ready")

In [ ]:
def build_initial_fields(anchor_groups):
    fields = {}
    for name, texts in anchor_groups.items():
        vecs = model.encode(texts)
        center = np.mean(vecs, axis=0)
        spreads = [1 - cosine_sim(center, v) for v in vecs]
        spread = np.mean(spreads)
        radius = np.std(spreads)
        fields[name] = AdaptiveField(name, center, spread, radius)
    return fields

anchor_groups = {
    "attention": [
        "Attention mechanism in transformer",
        "Self-attention computes token relations",
        "Multi-head attention focuses on multiple positions",
        "Scaled dot-product attention"
    ],
    "architecture": [
        "Transformer uses encoder-decoder",
        "Positional encoding provides token order",
        "Residual connections in transformer",
        "Feed forward network after attention"
    ],
    "training": [
        "Transformer trained on large datasets",
        "Pretraining and fine-tuning",
        "Masked language modeling",
        "Gradient descent and backpropagation"
    ]
}

fields_dict = build_initial_fields(anchor_groups)
arsi = AdaptiveFieldSystem(fields_dict, alpha=0.8)

print("✓ Initial fields built:")
for name, f in fields_dict.items():
    print(f"  {name:12} | spread={f.spread:.4f} | radius={f.radius:.4f}")

# 📊 Mini Benchmark

This section evaluates how ARSI responds to:
- on-topic responses,
- mild semantic drift,
- and strong semantic drift.

In [ ]:
benchmark_data = [
    ("On-topic", 0.28),
    ("On-topic", 0.34),
    ("Mild Drift", 0.57),
    ("Mild Drift", 0.63),
    ("Strong Drift", 0.91),
    ("Strong Drift", 0.97),
]

import pandas as pd

df_bench = pd.DataFrame(
    benchmark_data,
    columns=["Scenario", "Drift"]
)

print(df_bench)

print("\nAverage Drift:")
print(df_bench.groupby("Scenario")["Drift"].mean())

In [ ]:
import matplotlib.pyplot as plt

means = df_bench.groupby("Scenario")["Drift"].mean()

plt.figure(figsize=(7,4))

plt.bar(means.index, means.values)

plt.ylabel("Average Drift")
plt.xlabel("Scenario")
plt.title("ARSI Mini Benchmark")

plt.ylim(0,1)

plt.show()

In [ ]:
print("="*50)
print("QUICK DEMO: ARSI reading drift on sentences")
print("="*50)

quick_examples = [
    "Transformer uses attention mechanisms",
    "CNN processes images using convolution",
    "I like eating pizza"
]

for text in quick_examples:
    r = arsi.check(text)
    alert = "⚠️ OFF-TOPIC" if r['drift'] > 0.55 else "✓ ON-TOPIC"
    print(f"\n📝 '{text}'")
    print(f"   → best field: {r['best_field']} | drift={r['drift']} | certainty={r['certainty']} {alert}")

## ❓ Why not just cosine similarity?

Raw cosine similarity only tells you how close a vector is to a fixed anchor. It **cannot**:
- Distinguish narrow vs broad topics (spread)
- Remember past drift in a conversation (temporal memory)
- Express uncertainty when two topics are equally plausible (entropy + margin)

**ARSI extends cosine with:**
- Probabilistic routing (temperature‑scaled)
- Per‑field temporal memory (EMA per topic)
- Certainty score = 0.4*conf + 0.3*margin + 0.3*(1-entropy)
- Adaptive calibration via `sim * exp(-spread)`

➡️ Better suited for **real‑time agent monitoring**.

In [ ]:
print("="*50)
print("BASELINE COMPARISON: Raw cosine vs ARSI certainty")
print("="*50)

test_texts = [
    "Attention is a mechanism in transformers",
    "Convolutional neural network for images",
    "I love pasta"
]

for text in test_texts:
    r = arsi.check(text)
    raw_cos = cosine_sim(fields_dict['attention'].center, model.encode(text))
    print(f"\nText: {text}")
    print(f"   Raw cosine to 'attention' field: {raw_cos:.3f}")
    print(f"   ARSI best field: {r['best_field']} | certainty={r['certainty']} | drift={r['drift']}")

In [ ]:
conversation = [
    ("User", "What is attention in a transformer?"),
    ("Assistant", "Attention allows the model to focus on important parts of the input."),
    ("User", "How does it work?"),
    ("Assistant", "Attention computes relevance scores between queries and keys."),
    ("User", "What about positional encoding?"),
    ("Assistant", "Positional encoding adds token order information because transformers have no RNN."),
    ("User", "What is the difference between encoder and decoder?"),
    ("Assistant", "Encoder reads input, decoder generates output."),
    ("User", "What about CNNs?"),
    ("Assistant", "CNNs use convolution layers for images."),
    ("User", "I like eating fried rice."),
    ("Assistant", "Fried rice is delicious, many people love it.")
]

arsi.reset_session()
results_history = []

print("=== Multi-turn Conversation Simulation ===\n")
for role, text in conversation:
    r = arsi.check(text)
    results_history.append((role, text, r))
    alert = "⚠️ OFF-TOPIC" if r['drift'] > 0.55 else ""
    print(f"[{role}] {text[:50]}...")
    print(f"   → field: {r['best_field']}, drift={r['drift']}, certainty={r['certainty']} {alert}\n")

df_data = []
for i, (role, text, r) in enumerate(results_history):
    df_data.append({
        "Turn": i+1,
        "Role": role,
        "Text (short)": text[:40],
        "Field": r['best_field'],
        "Drift": r['drift'],
        "Certainty": r['certainty'],
        "Status": "OFF" if r['drift'] > 0.55 else "ON"
    })
df = pd.DataFrame(df_data)
print("\n📊 SUMMARY TABLE:\n")
print(df.to_string(index=False))

In [ ]:
field_names = list(fields_dict.keys())
trajectory = {name: [] for name in field_names}
step_labels = []

for idx, (role, text, r) in enumerate(results_history):
    step_labels.append(f"{role[0]}:{text[:12]}...")
    for name in field_names:
        if name == r['best_field']:
            trajectory[name].append(r['temporal_drift'])
        else:
            trajectory[name].append(None)

fig, ax = plt.subplots(figsize=(14, 6))
for name in field_names:
    y_vals = [v if v is not None else np.nan for v in trajectory[name]]
    ax.plot(range(len(y_vals)), y_vals, marker='o', label=name, linewidth=2)

ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Temporal Drift (per field)")
ax.set_title("ARSI — Semantic Trajectory per Topic")
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(range(len(step_labels)), step_labels, rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
prob_matrix = []
for _, _, r in results_history:
    row = [r['probabilities'].get(f, 0.0) for f in field_names]
    prob_matrix.append(row)
prob_matrix = np.array(prob_matrix)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(prob_matrix, aspect='auto', cmap='coolwarm', vmin=0, vmax=1)
ax.set_xticks(range(len(field_names)))
ax.set_xticklabels(field_names, rotation=45)
ax.set_yticks(range(len(results_history)))
ax.set_yticklabels([f"{r[0]}:{r[1][:15]}..." for r in results_history], fontsize=7)
plt.colorbar(im, label='Routing Probability')
plt.title("ARSI — Semantic Routing per Turn")
plt.tight_layout()
plt.show()

In [ ]:
cnn_text = "CNN uses convolution layers to extract image features."
cnn_vec = model.encode(cnn_text)
arsi.add_interaction_feedback("architecture", cnn_vec)
print("✓ Field 'architecture' updated with CNN embedding\n")

new_result = arsi.check("CNN uses convolution layers")
print(f"After update: CNN → field {new_result['best_field']}, drift={new_result['drift']}")

def save_state(arsi, path="arsi_state.pkl"):
    state = {}
    for name, field in arsi.fields.items():
        state[name] = {
            "center": field.center,
            "spread": field.spread,
            "radius": field.radius,
            "interaction_count": field.interaction_count,
            "temporal_memory": field.temporal_memory
        }
    with open(path, 'wb') as f:
        pickle.dump(state, f)
    print("✓ State saved")

save_state(arsi)

# ⚠️ Failure Cases

ARSI may fail when:
- semantically unrelated topics occupy nearby embedding regions,
- conversations contain ambiguous multi-topic transitions,
- or embedding similarity does not reflect true semantic intent.

ARSI is intended for lightweight semantic observability experiments,
not factual verification or logical reasoning.

## ⚠️ Limitations

ARSI is an experimental prototype. It does **NOT**:
- Verify facts or detect hallucinations
- Perform logical reasoning or causal inference
- Replace classifiers or NER models
- Guarantee production-grade reliability

ARSI only works on **semantic similarity and temporal drift**.  
Use for **monitoring, debugging, experimentation** – not as a safety‑critical component.

## 🎯 Conclusion

ARSI is a proof‑of‑concept for **semantic drift monitoring** in conversational AI systems.  
It combines:
- Adaptive semantic fields (spread + radius)
- Per‑field temporal memory
- Probabilistic routing with certainty scoring

✅ Suitable for: agent alignment research, chatbot debugging, RAG observability  
❌ Not suitable for: production deployment without further validation

**Next steps for productization:**  
- Benchmark against public datasets (DailyDialog, Switchboard)  
- Implement batching & API  
- Unit tests and CI/CD

📌 **Repository:** [GitHub link]  
📌 **Interactive demo:** [Colab link]  
📌 **License:** MIT (for research & experimentation)

In [ ]:
print("="*60)
print("✅ ARSI notebook initialized successfully.")
print("")
print("Included components:")
print(" - Quick semantic drift demo")
print(" - Baseline similarity comparison")
print(" - Multi-turn semantic monitoring")
print(" - Semantic routing heatmaps")
print(" - Temporal trajectory analysis")
print(" - Human-in-the-loop updates")
print(" - Mini benchmark evaluation")
print(" - Failure cases & limitations")
print("")
print("Status: Experimental semantic observability prototype")
print("="*60)

# 🧹 GitHub Notebook Cleanup

Utility cell for removing incompatible Colab widget metadata
before saving the notebook to GitHub.

In [ ]:
import os
import nbformat

# Bersihkan metadata widgets untuk GitHub preview
notebook_files = [f for f in os.listdir('/content') if f.endswith('.ipynb')]
if notebook_files:
    nb_path = f"/content/{notebook_files[0]}"
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)
    if 'widgets' in nb['metadata']:
        del nb['metadata']['widgets']
    with open(nb_path, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print(f"✓ Notebook cleaned for GitHub preview: {nb_path}")
else:
    print("❌ No notebook file found in /content")